# RAG 파이프라인 구축하기

## 1. 사전작업(Pre-processing) - 1~4 단계

사전 작업 단계에서는 데이터 소스를 Vector DB (저장소) 에 문서를 로드-분할-임베딩-저장 하는 4단계를 진행합니다.

- 1단계 문서로드(Document Load): 문서 내용을 불러옵니다.
- 2단계 분할(Text Split): 문서를 특정 기준(Chunk) 으로 분할합니다.
- 3단계 임베딩(Embedding): 분할된(Chunk) 를 임베딩하여 저장합니다.
- 4단계 벡터DB 저장: 임베딩된 Chunk 를 DB에 저장합니다.

## 2. RAG 수행(RunTime) - 5~8 단계

- 5단계 검색기(Retriever): 쿼리(Query) 를 바탕으로 DB에서 검색하여 결과를 가져오기 위하여 리트리버를 정의합니다. 리트리버는 검색 알고리즘이며(Dense, Sparse) 리트리버로 나뉘게 됩니다. Dense: 유사도 기반 검색, Sparse: 키워드 기반 검색
- 6단계 프롬프트: RAG 를 수행하기 위한 프롬프트를 생성합니다. 프롬프트의 context 에는 문서에서 검색된 내용이 입력됩니다. 프롬프트 엔지니어링을 통하여 답변의 형식을 지정할 수 있습니다.
- 7단계 LLM: 모델을 정의합니다.(GPT-3.5, GPT-4, Claude, etc..)
- 8단계 Chain: 프롬프트 - LLM - 출력 에 이르는 체인을 생성합니다.

## * 실습에 활용할 문서

각자 1차 과제때 구상한 서비스를 구현하기 위해 필요한 문서를 구비하여 업로드 해주세요.

예시문서
- 문서명: <간이투자설명서> 삼성KODEX 코스피증권상장지수투자신탁[주식](펀드 코드: B6290)
- 링크: https://www.samsungfund.com/etf/product/library/pdf.do
- 파일명: `2ETF52-Aa`

_실습을 위해 다운로드 받은 파일을 구글 Colab의 `/content` 폴더로 복사해 주시기 바랍니다_

## * 뼈대 코드 활용

아래는 기본적인 RAG 구조 이해를 위한 뼈대코드(skeleton code) 입니다. 각 단계별 모듈의 내용을 앞으로 상황에 맞게 변경하면서 문서에 적합한 구조를 찾아갈 수 있습니다.(각 단계별로 다양한 옵션을 설정하거나 새로운 기법을 적용할 수 있습니다.)

In [ ]:
## 환경설정 : 필요한 라이브러리들을 설정합니다.

!pip install python-dotenv
!pip install faiss-cpu
!pip install pymupdf
!pip install langchain_community
!pip install langchain_openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 3.2 MB/s eta 0:00:00


In [9]:
# 아래를 참고하여 생성한 API 키를 환경변수로 관리하기 위한 .txt 파일을 만들어 보세요
# txt 파일에 OPENAI_API_KEY=your-openai-api-key-here(발급받은 api key)를 붙혀넣기 하고.'env'라는 이름으로 저장한 뒤, /content에 파일을 업로드 해주세요

from dotenv import load_dotenv
import openai
import os

# API 키 정보 로드
load_dotenv()

# .env파일 생성 및 저장
api_key = os.getenv("OPENAI_API_KEY")
openai.api_key = api_key

In [ ]:
## RAG 기본 파이프라인(1~8단계)

#langchain 라이브러리에서 필요한 기능들을 불러와 줍니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
# 단계 1: 문서 로드(Load Documents)
# 구현하고자 하는 서비스에 맞게, 직접 구비한 문서를 업로드 해주세요. .꼭 .pdf파일이 아니어도 좋습니다.
loader = PyMuPDFLoader("/content/2ETF52-Aa.pdf")
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

"""페이지의 내용을 출력합니다."""

print(docs[1].page_content)

"""`metadata` 를 확인합니다."""

docs[1].__dict__

문서의 페이지수: 3
2
운용전문
인력
성명
생년
직위
운용현황
동종집합투자기구
연평균 수익률(주식)
운용
경력년수
집합투자
기구 수
운용규모
운용역
운용사
최근1년
최근2년
최근1년
최근2년
이대환
1988
책임(VP)
21개
  30,754억원
2.71%
17.36%
0.63%
9.28%
5년 6개월
-
상기인은 이 투자신탁의 투자전략 수립 및 투자의사결정 등에 주도적·핵심적 역할을 수행하는 운용전문인력입니다.
-
운용중인 다른 집합투자기구 중 성과보수가 약정된 집합투자기구 :
해당사항없음
-
위의 집합투자기구 수에는 모자형투자신탁의 모투자신탁은 포함되어 있지 않습니다.
-
운용전문인력의 최근 과거 3년 이내에 운용한 집합투자기구의 명칭, 집합투자재산의 규모와 수익률 등은 금융투자협회 홈페이지(www.kofia.or.kr)에서 확인
할수 있습니다.
-
동종집합투자기구 연평균 수익률은 해당 집합투자업자가 분류한 동일 유형 집합투자기구의 평균운용성과이며, 해당 운용전문인력의 평균운용성과는 해당 회
사근무기간 중 운용한 성과를 의미합니다.
-
'운용경력년수'는 해당 운용전문인력이 과거 집합투자기구를 운용한 기간을 모두 합산한 기간입니다.
투자자
유의사항
·집합투자증권은 『예금자보호법』에 따라 보호되지 않는 실적배당상품으로 투자원금의 손실이 발생할 수 있으므로 투자에
  신중을 가하여 주시기 바랍니다.
·금융위원회가 투자설명서의 기재사항이 진실 또는 정확하다는 것을 인정하거나 그 증권의 가치를 보증 또는 승인하지 아니
  함을 유의하시기 바랍니다.
·간이투자설명서보다 자세한 내용이 필요하시어 투자설명서의 교부를 요청하시면 귀하의 집합투자증권 매입 이전까지 교부
  하오니 참고하시기 바랍니다.
·간이투자설명서는 증권신고서 효력발생일까지 기재사항 중 일부가 변경될 수 있으며, 개방형 집합투자증권인 경우 효력발
  생일 이후에도 변경될 수 있습니다.
·이 집합투자증권의 투자위험등급을 확인하시고, 귀하의 투자 경력이나 투자성향에 적합한 상품인지 신중한 투자결정을 하
  

{'id': None,
 'metadata': {'producer': 'iText 4.2.0 by 1T3XT',
  'creator': '',
  'creationdate': '2025-01-22T13:49:42+09:00',
  'source': '/content/2ETF52-Aa.pdf',
  'file_path': '/content/2ETF52-Aa.pdf',
  'total_pages': 3,
  'format': 'PDF 1.4',
  'title': '',
  'author': '',
  'subject': '',
  'keywords': '',
  'moddate': '2025-01-22T13:49:42+09:00',
  'trapped': '',
  'modDate': "D:20250122134942+09'00'",
  'creationDate': "D:20250122134942+09'00'",
  'page': 1},
 'page_content': "2\n운용전문\n인력\n성명\n생년\n직위\n운용현황\n동종집합투자기구\n연평균 수익률(주식)\n운용\n경력년수\n집합투자\n기구 수\n운용규모\n운용역\n운용사\n최근1년\n최근2년\n최근1년\n최근2년\n이대환\n1988\n책임(VP)\n21개\n  30,754억원\n2.71%\n17.36%\n0.63%\n9.28%\n5년 6개월\n-\n상기인은 이 투자신탁의 투자전략 수립 및 투자의사결정 등에 주도적·핵심적 역할을 수행하는 운용전문인력입니다.\n-\n운용중인 다른 집합투자기구 중 성과보수가 약정된 집합투자기구 :\n해당사항없음\n-\n위의 집합투자기구 수에는 모자형투자신탁의 모투자신탁은 포함되어 있지 않습니다.\n-\n운용전문인력의 최근 과거 3년 이내에 운용한 집합투자기구의 명칭, 집합투자재산의 규모와 수익률 등은 금융투자협회 홈페이지(www.kofia.or.kr)에서 확인\n할수 있습니다.\n-\n동종집합투자기구 연평균 수익률은 해당 집합투자업자가 분류한 동일 유형 집합투자기구의 평균운용성

In [ ]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

분할된 청크의수: 14


In [10]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

In [13]:
# 단계 4: DB 생성(Create DB) 및 저장
import faiss

# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

for doc in vectorstore.similarity_search("투자"):
    print(doc.page_content)

시기 바랍니다.
·증권신고서, 투자설명서상 기재된 투자전략에 따른 투자목적 또는 성과목표는 반드시 실현된다는 보장은 없습니다. 또한
  과거의 투자실적이 장래에도 실현된다는 보장은 없습니다.
·집합투자기구가 설정 후 1년이 경과하였음에도 설정액이 50억원 미만(소규모펀드)인 경우 분산투자가 어려워 효율적인
  자산운용이 곤란하거나 임의해지 될수 있으니, 투자 시 소규모펀드 여부를 확인 하시기 바라며 소규모펀드 해당여부는
  금융투자협회, 판매회사, 자산운용사 홈페이지에서 확인할 수 있습니다.
구분
투자위험의 주요내용
주요투자
위험
1
주식등 가격변동
위험
이 투자신탁은 증권시장에 상장되어 거래되는 주식 등에 주로 투자하기 때문에 동 주식의 가격변동으로 인
한 손실위험에 노출됩니다. 또한, 투자신탁재산의 가치는 투자대상종목 발행회사의 영업환경, 재무상황 및
신용상태의 악화에 따라 급격히 변동될 수 있습니다.
1
추적오차
발생위험
보가 필요하신 경우에는 동 집합투자증권을 매입하기 이전에 투자설명서를 반드시 참고하시기 바랍니다.
[요약정보]
투자목적
및
투자전략
이 투자신탁은 국내주식을 주된 투자대상으로 하여 신탁재산의 60% 이상을 투자하며, 유가증권시장에 상장된 종목을 기초로 시가총액비중으로
가중하여 산출되는 '코스피 지수'를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운용할 계
획입니다.
분류
투자신탁, 증권(주식형), 개방형(중도환매가능), 추가형(추가납입가능), 상장지수투자신탁
투자비용
명칭
투자자가 부담하는 수수료, 총보수 및 비용
1,000만원 투자시 투자자가 부담하는 투자기간
별 총보수ㆍ비용 예시 (단위 : 천원)
판매수수료
총보수
동종유형
총보수
총 보수
ㆍ비용
1년
2년
3년
5년
10년
판매보수
삼성KODEX 코스피증권상장
지수투자신탁[주식]
없음
 0.1500%
 0.001%
 0.1639%
    16
    33
    51
    89
신용상태의 악화에 따라 급격히 변동될 수 있습니다.
1
추

In [14]:
# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

"""검색기에 쿼리를 날려 검색된 chunk 결과를 확인합니다."""

# 검색기에 쿼리를 날려 검색된 chunk 결과를 확인합니다.
retriever.invoke("투자자 유의사항은?")

[Document(id='4cd83400-0937-4d94-83d5-928a6be9cb9b', metadata={'producer': 'iText 4.2.0 by 1T3XT', 'creator': '', 'creationdate': '2025-01-22T13:49:42+09:00', 'source': '/content/2ETF52-Aa.pdf', 'file_path': '/content/2ETF52-Aa.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-22T13:49:42+09:00', 'trapped': '', 'modDate': "D:20250122134942+09'00'", 'creationDate': "D:20250122134942+09'00'", 'page': 1}, page_content='시기 바랍니다.\n·증권신고서, 투자설명서상 기재된 투자전략에 따른 투자목적 또는 성과목표는 반드시 실현된다는 보장은 없습니다. 또한\n  과거의 투자실적이 장래에도 실현된다는 보장은 없습니다.\n·집합투자기구가 설정 후 1년이 경과하였음에도 설정액이 50억원 미만(소규모펀드)인 경우 분산투자가 어려워 효율적인\n  자산운용이 곤란하거나 임의해지 될수 있으니, 투자 시 소규모펀드 여부를 확인 하시기 바라며 소규모펀드 해당여부는\n  금융투자협회, 판매회사, 자산운용사 홈페이지에서 확인할 수 있습니다.\n구분\n투자위험의 주요내용\n주요투자\n위험\n1\n주식등 가격변동\n위험\n이 투자신탁은 증권시장에 상장되어 거래되는 주식 등에 주로 투자하기 때문에 동 주식의 가격변동으로 인\n한 손실위험에 노출됩니다. 또한, 투자신탁재산의 가치는 투자대상종목 발행회사의 영업환경, 재무상황 및\n신용상태의 악화에 따라 급격히 변동될 수 있습니다.\n1\n추적오차\n발생위험'),
 Document(id='563

In [15]:
# 단계 6: 프롬프트 입력
# 2주차 과제에서 생성한 프롬프트를 여기에 입력해 주세요. 2주차 과제 피드백을 보고 zero-shot, few-shot, 후카츠식 프롬프트, 동적(dynamic) 프롬프팅 등 다양한 프롬프팅 기법을 적용해 보세요.
prompt = PromptTemplate.from_template(
    """You are a financial advisor chatbot.
Provide information about financial products, investment strategies,
and personal finance management.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

In [16]:
# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0) #2주차 과제와 마찬가지로 여러가지 모델을 연동할 수 있습니다. OpenAI뿐 아니라 Gemini나 Claude 등 다양한 모델을 연동해 보셔도 좋습니다.

# 단계 8: 체인(Chain) 생성
# promt, llm, vectodDB를 이어붙인 간단한 chain입니다. 제공드린 핵심예제를 보고 기능에 맞게 추가로 다른 체인을 연동해 보세요.
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

"""생성된 체인에 쿼리(질문)을 입력하고 실행합니다."""

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "투자자 유의사항은?"
response = chain.invoke(question)
print(response)

투자자 유의사항은 다음과 같습니다:

1. 집합투자증권은 『예금자보호법』에 따라 보호되지 않는 실적배당상품으로, 투자원금의 손실이 발생할 수 있으므로 투자에 신중을 기해야 합니다.
2. 금융위원회가 투자설명서의 기재사항이 진실 또는 정확하다는 것을 인정하거나 그 증권의 가치를 보증 또는 승인하지 않음을 유의해야 합니다.
3. 간이투자설명서보다 자세한 내용이 필요할 경우, 투자설명서의 교부를 요청할 수 있으며, 이는 집합투자증권 매입 이전까지 교부됩니다.
4. 간이투자설명서는 증권신고서 효력발생일까지 기재사항 중 일부가 변경될 수 있으며, 개방형 집합투자증권인 경우 효력발생일 이후에도 변경될 수 있습니다.
5. 집합투자증권의 투자위험등급을 확인하고, 자신의 투자 경력이나 투자성향에 적합한 상품인지 신중한 투자결정을 해야 합니다.
